[Guide Index](README.md)

# 02.04 Reading and Building Graph Content

This guide covers the methods for reading and building the content of a graph.

1. RDF terms and Python values
2. RDF lists
3. single values
4. walks
5. prefixes
6. what a graph contains


## How to run this notebook

This guide assumes StarLayer has been pip installed — see [Getting Started](01.01-getting-started.ipynb). Run the setup cell below first, then the sections you want, from top to bottom within each section. Each section builds the data it needs; the setup cell defines the names they share.

In [1]:
from starlayer.graph import NamedNode, Namespace, StarLayerGraph, literal

EXL = Namespace("http://example.org/")

## 02.04.1 Terms and Python values

A graph holds RDF terms. In StarLayer an RDF term is a pyoxigraph object (`NamedNode`, `BlankNode`, `Literal` or `Triple` from the pyoxigraph library). Python code usually works with Python values, and text is a Python value too: a `str`. This section covers how to convert between these two: a Python value to an RDF term, an RDF term to a Python value, and Turtle text (a `str` written in Turtle syntax) to an RDF term. These words are used throughout the section:

| Word | In StarLayer | Meaning |
| --- | --- | --- |
| RDF term | a pyoxigraph `NamedNode`, `BlankNode`, `Literal` or `Triple` | an IRI, a blank node, a literal or a triple term |
| IRI | pyoxigraph `NamedNode` | the name of a resource, such as `http://example.org/a` |
| blank node | pyoxigraph `BlankNode` | a resource with no name |
| literal | pyoxigraph `Literal`, with `.value` (the lexical form), `.datatype` and `.language` | a value written in the data: text with a datatype or, for text, a language. In `"4"^^xsd:integer` the lexical form, the text between the quotes, is `4` and the datatype is `xsd:integer` |
| triple | Python `tuple` `(s, p, o)` | a subject, a predicate and an object, each an RDF term |
| triple term | pyoxigraph `Triple` | a whole triple used as a value (RDF 1.2) |
| Python value | a built-in Python object: `int`, `bool`, `float`, `str`, `Decimal`, `date` and so on | an ordinary Python value, not an RDF term |
| Turtle text | Python `str` | text in Turtle syntax, such as `<http://example.org/a>` or `"hej"@sv` |

The four pyoxigraph classes can be imported from `starlayer.graph`. The pyoxigraph terms are Python objects too, but in this section a *Python value* means the built-in kinds in the table, and the pyoxigraph classes are called RDF terms. A triple, a Python value and Turtle text are ordinary Python objects, not pyoxigraph objects.

Three functions in `starlayer.graph` convert between them:

| Function | Converts | To |
| --- | --- | --- |
| `literal(value, datatype=None, lang=None, direction=None)` | a Python value, or the lexical form of a literal | a pyoxigraph `Literal` |
| `to_python(term)` | a pyoxigraph `Literal` | a Python value |
| `parse_term(text, prefixes=None)` | Turtle text | an RDF term |

The three subsections below take them in turn, and the last one shows what each does when its input is wrong.

### 02.04.1.1 A Python value to a pyoxigraph Literal: `literal()`

`literal(value)` makes a pyoxigraph `Literal` from a Python value. The type of the value chooses the datatype:

| Python value | Datatype of the pyoxigraph `Literal` |
| --- | --- |
| `bool` | `xsd:boolean` |
| `int` | `xsd:integer` |
| `float` | `xsd:double` |
| `Decimal` | `xsd:decimal` |
| `date`, `datetime`, `time` | `xsd:date`, `xsd:dateTime`, `xsd:time` |
| `str` | `xsd:string` (a plain string) |

A `str` can also be the lexical form of a literal: to give it a datatype, pass both, as in `literal("4", XSD.integer)`. `lang` makes a language-tagged literal (`"hej"@sv`), and `direction="ltr"` or `"rtl"` (RDF 1.2, needs `lang`) adds a base direction. A literal has a language or a datatype, not both: `literal("5", XSD.integer, lang="sv")` raises `ValueError`.

Below, a Python value of each type is made into a pyoxigraph `Literal`, the parts of a pyoxigraph `Literal` are shown, and then a lexical form is given a datatype, a language, and a language with a direction.

In [2]:
import datetime
from decimal import Decimal

from starlayer.graph import XSD, literal

print("1. A Python value to a pyoxigraph Literal")
for value in (True, 5, 2.5, Decimal("1.50"), datetime.date(2026, 1, 2), "text"):
    print(repr(value), "->", literal(value))

print("\n2. The parts of a pyoxigraph Literal")
five = literal(5)
print("lexical form:", five.value, "| datatype:", five.datatype, "| language:", five.language)

print("\n3. A lexical form with a datatype, a language, and a language with a direction")
print(literal("4", XSD.integer), literal("hej", lang="sv"), literal("مرحبا", lang="ar", direction="rtl"))

1. A Python value to a pyoxigraph Literal
True -> "true"^^<http://www.w3.org/2001/XMLSchema#boolean>
5 -> "5"^^<http://www.w3.org/2001/XMLSchema#integer>
2.5 -> "2.5"^^<http://www.w3.org/2001/XMLSchema#double>
Decimal('1.50') -> "1.50"^^<http://www.w3.org/2001/XMLSchema#decimal>
datetime.date(2026, 1, 2) -> "2026-01-02"^^<http://www.w3.org/2001/XMLSchema#date>
'text' -> "text"

2. The parts of a pyoxigraph Literal
lexical form: 5 | datatype: <http://www.w3.org/2001/XMLSchema#integer> | language: None

3. A lexical form with a datatype, a language, and a language with a direction
"4"^^<http://www.w3.org/2001/XMLSchema#integer> "hej"@sv "مرحبا"@ar--rtl


### 02.04.1.2 A pyoxigraph Literal to a Python value: `to_python()`

`to_python(term)` makes a Python value from a pyoxigraph `Literal`. The `.datatype` of the pyoxigraph `Literal` chooses the type (`int`, `float`, `Decimal`, `bool`, `date`, `datetime`, `time`, or `str` for a string, a language-tagged string, or a datatype with no Python equivalent), and the `.value`, the lexical form, builds the value. A language-tagged string loses its language and direction in the Python value, and the pyoxigraph `Literal` itself keeps them.

Below, each Python value from the previous section goes to a pyoxigraph `Literal` and back, and then a language-tagged literal shows what is lost.

In [ ]:
from starlayer.graph import to_python

print("1. Python value -> pyoxigraph Literal -> Python value")
for value in (True, 5, 2.5, Decimal("1.50"), datetime.date(2026, 1, 2), "text"):
    term = literal(value)
    print(repr(value), "->", term, "->", repr(to_python(term)))

print("\n2. A language-tagged literal")
hej = literal("hej", lang="sv")
print("to_python drops the language:", repr(to_python(hej)), "| the pyoxigraph Literal keeps it:", hej.language)

1. Python value -> pyoxigraph Literal -> Python value
True -> "true"^^<http://www.w3.org/2001/XMLSchema#boolean> -> True
5 -> "5"^^<http://www.w3.org/2001/XMLSchema#integer> -> 5
2.5 -> "2.5"^^<http://www.w3.org/2001/XMLSchema#double> -> 2.5
Decimal('1.50') -> "1.50"^^<http://www.w3.org/2001/XMLSchema#decimal> -> Decimal('1.50')
datetime.date(2026, 1, 2) -> "2026-01-02"^^<http://www.w3.org/2001/XMLSchema#date> -> datetime.date(2026, 1, 2)
'text' -> "text" -> 'text'

2. A language-tagged literal
to_python drops the language: 'hej' | the pyoxigraph Literal keeps it: sv


### 02.04.1.3 Turtle text to an RDF term: `parse_term()`

`parse_term(text, prefixes=None)` reads one RDF term written as Turtle text and returns the pyoxigraph `NamedNode`, pyoxigraph `BlankNode`, pyoxigraph `Literal` or pyoxigraph `Triple` it stands for. A prefixed name such as `ex:a` needs `prefixes`. The text must be Turtle, so a lexical form on its own (`abc`) is an error, and `4` is Turtle for an `xsd:integer`. To give a lexical form a datatype, use `literal("4", datatype)`. Turtle writes the terms like this:

| Turtle text | Returns |
| --- | --- |
| `<http://example.org/a>` | a pyoxigraph `NamedNode` |
| `ex:a`, with `prefixes={"ex": "http://example.org/"}` | the same pyoxigraph `NamedNode` |
| `_:b1` | a pyoxigraph `BlankNode` |
| `<<( <urn:s> <urn:p> <urn:o> )>>` | a pyoxigraph `Triple`, a triple term |
| `"4"` | a pyoxigraph `Literal`: lexical form `4`, datatype `xsd:string` |
| `"4"^^xsd:integer` | a pyoxigraph `Literal` with datatype `xsd:integer` |
| `4`, `4.5`, `4e1`, `true` | a pyoxigraph `Literal` with datatype `xsd:integer`, `xsd:decimal`, `xsd:double`, `xsd:boolean` |
| `"hej"@sv` | a pyoxigraph `Literal` with language `sv` (a tag is lowercased: `@en-US` is `en-us`) |
| `"مرحبا"@ar--rtl` | a pyoxigraph `Literal` with language `ar` and base direction `rtl` |

Below, each of those forms is read.

In [ ]:
from starlayer.graph import parse_term

print("1. Literals: a literal is typed by how it is written")
for text in ('"4"', '"4"^^xsd:integer', "4", "4.5", "4e1", "true", '"hej"@sv', '"مرحبا"@ar--rtl'):
    print(f"{text:18}", "->", parse_term(text, {"xsd": str(XSD)}))

print("\n2. An IRI and a triple term")
print(parse_term("<http://example.org/a>"), parse_term("<<( <urn:s> <urn:p> <urn:o> )>>"))

1. Literals: a literal is typed by how it is written
"4"                -> "4"
"4"^^xsd:integer   -> "4"^^<http://www.w3.org/2001/XMLSchema#integer>
4                  -> "4"^^<http://www.w3.org/2001/XMLSchema#integer>
4.5                -> "4.5"^^<http://www.w3.org/2001/XMLSchema#decimal>
4e1                -> "4e1"^^<http://www.w3.org/2001/XMLSchema#double>
true               -> "true"^^<http://www.w3.org/2001/XMLSchema#boolean>
"hej"@sv           -> "hej"@sv
"مرحبا"@ar--rtl    -> "مرحبا"@ar--rtl

2. An IRI and a triple term
<http://example.org/a> <urn:s> <urn:p> <urn:o>


### 02.04.1.4 When the input is wrong

Each function refuses input that it cannot convert:

| Call | Problem | Raises |
| --- | --- | --- |
| `to_python(term)` | the lexical form is not valid for the datatype (`abc` for `xsd:integer`) | `FormatError`, kind `"term"` |
| `to_python(term)` | the term is a pyoxigraph `NamedNode` or pyoxigraph `BlankNode` | `TypeError` |
| `parse_term(text)` | the text is not exactly one RDF term | `FormatError`, kind `"term"` |
| `parse_term(text)` | the argument is not text | `TypeError` |
| `literal(...)` | a language and a datatype together, or a direction without a language | `ValueError` |

`parse_term` keeps a lexical form that is not valid for its datatype (`"abc"^^xsd:integer`) as written, and `to_python` is the call that refuses it.

Below, each problem from the table is raised and the error is printed.

In [ ]:
from starlayer.errors import FormatError

bad_integer = parse_term('"abc"^^xsd:integer', {"xsd": str(XSD)})
print("parse_term keeps the lexical form as written:", bad_integer, "\n")

calls = (
    ("to_python, a bad lexical form", lambda: to_python(bad_integer)),
    ("to_python, an IRI", lambda: to_python(parse_term("<http://example.org/a>"))),
    ("parse_term, two terms", lambda: parse_term("4 5")),
    ("parse_term, not text", lambda: parse_term(5)),
    ("literal, a language and a datatype", lambda: literal("5", XSD.integer, lang="sv")),
)
for label, call in calls:
    try:
        call()
    except (FormatError, TypeError, ValueError) as error:
        kind = f" (kind {error.kind})" if isinstance(error, FormatError) else ""
        print(f"{label:36} {type(error).__name__}{kind}: {error}")

parse_term keeps the lexical form as written: "abc"^^<http://www.w3.org/2001/XMLSchema#integer> 

to_python, a bad lexical form        FormatError (kind term): 'abc' is not a valid integer
to_python, an IRI                    TypeError: only a literal has a Python value, not NamedNode
parse_term, two terms                FormatError (kind term): not a term: '4 5'
parse_term, not text                 TypeError: parse_term() needs a str, not int
literal, a language and a datatype   ValueError: a literal has a language tag or a datatype, not both: lang='sv' with datatype <http://www.w3.org/2001/XMLSchema#integer>


## 02.04.2 Lists

### How an RDF list is stored

RDF has no inherent list type. A list is a chain of list nodes. Each list node is a blank node with two triples: `rdf:first` gives the member and `rdf:rest` gives the next list node (a blank node), or `rdf:nil` at the end of the list. The list is named by its head, the first list node, and an ordinary triple points at the head (`ex:order ex:steps _:node1`).

Turtle writes the chain as `( ex:fetch ex:parse 3 )`. StarLayer reads that into list nodes when it parses, and `serialize(pretty=True)` writes the list nodes back as `( ... )`. Without `pretty`, they come out as ordinary triples.

In [3]:
from starlayer.graph import RDF, BlankNode

cg = StarLayerGraph(prefixes={"ex": "http://example.org/", "rdf": str(RDF)})
cg.parse(data="@prefix ex: <http://example.org/> . ex:order ex:steps ( ex:fetch ex:parse 3 ) .", format="turtle")

# A blank node has no name, so label each list node (_:node1, _:node2, ...) in the order we meet it.
labels = {}

def show(term):
    """Short text for a term: a label for a blank node, a prefixed name for anything else."""
    if isinstance(term, BlankNode):
        labels.setdefault(term, f"_:node{len(labels) + 1}")
        return labels[term]
    return cg.qname_term(term)

print("1. The triple that points at the head of the list")
head = cg.value(EXL.order, EXL.steps)
print("  ", show(EXL.order), show(EXL.steps), show(head))

print("\n2. The list nodes, from the head to rdf:nil")
node = head
while node != RDF.nil:
    member = cg.value(node, RDF.first)
    next_node = cg.value(node, RDF.rest)
    print("  ", show(node), "rdf:first", show(member))
    print("  ", show(node), "rdf:rest ", show(next_node))
    node = next_node

print("\n3. The same list written back as Turtle")
print(cg.serialize(format="turtle", only_used=True, pretty=True))

print("4. The empty list is rdf:nil, with no list nodes")
print("  ", show(RDF.nil), "->", list(cg.items(RDF.nil)))

1. The triple that points at the head of the list
   ex:order ex:steps _:node1

2. The list nodes, from the head to rdf:nil
   _:node1 rdf:first ex:fetch
   _:node1 rdf:rest  _:node2
   _:node2 rdf:first ex:parse
   _:node2 rdf:rest  _:node3
   _:node3 rdf:first "3"^^<http://www.w3.org/2001/XMLSchema#integer>
   _:node3 rdf:rest  rdf:nil

3. The same list written back as Turtle
@prefix ex: <http://example.org/> .

ex:order ex:steps ( ex:fetch ex:parse 3 ) .

4. The empty list is rdf:nil, with no list nodes
   rdf:nil -> []


### Reading and maintaining lists

Keeping a list by hand means adding and removing several triples in the right order. These methods make it easier to work with lists:

- `graph.items(head)` reads a list and returns its members as RDF terms. The empty list gives no items. It raises a `FormatError` of kind `"rdf"` for a list that is not well formed: a cycle, or a list node without exactly one `rdf:first` and one `rdf:rest`.
- `graph.add_list(items)` writes the list nodes for those RDF terms, on new blank nodes, and returns the head (`rdf:nil` for no items). Nothing points at the head yet: add that triple yourself, or use `set_list`.
- `graph.set_list(subject, predicate, items)` makes the items the one list that a subject has for a predicate. It removes the list nodes of the old list, writes the new one and links it to the subject.
- `graph.remove_list(head)` removes the list nodes of a list. The members and the triple that points at the head stay.

In [4]:
lg = StarLayerGraph(prefixes={"ex": "http://example.org/"})

def members():
    """The members of the list that ex:order has for ex:steps, as a Python list of strings."""
    head = lg.value(EXL.order, EXL.steps)  # the head: the object of the triple that points at the list
    return [str(member) for member in lg.items(head)]  # items() gives the members as RDF terms, and str() turns each into text

print("1. add_list writes the list nodes and returns the head. A triple has to point at the head")
rdf_terms = [EXL.fetch, EXL.parse, literal(3)]  # the members must be RDF terms: two IRIs and a Literal made from the Python integer 3
head = lg.add_list(rdf_terms)
lg.add((EXL.order, EXL.steps, head))
print("  members:", members(), "| triples in the graph:", len(lg))

print("\n2. remove_list removes the list nodes, but not the members or the triple that points at the head")
lg.remove_list(head)
print(lg.serialize(format="turtle", only_used=True, bnode_labels="short"))  # only the triple that points at the head is left, and it points at nothing now

print("\n3. set_list replaces the list in one call: it removes the old list nodes, writes the new list and points at it")
lg.set_list(EXL.order, EXL.steps, [EXL.parse, EXL.fetch])
print(lg.serialize(format="turtle", only_used=True, pretty=True))

1. add_list writes the list nodes and returns the head. A triple has to point at the head
  members: ['<http://example.org/fetch>', '<http://example.org/parse>', '"3"^^<http://www.w3.org/2001/XMLSchema#integer>'] | triples in the graph: 7

2. remove_list removes the list nodes, but not the members or the triple that points at the head
@prefix ex: <http://example.org/> .
ex:order ex:steps _:b0 .


3. set_list replaces the list in one call: it removes the old list nodes, writes the new list and points at it
@prefix ex: <http://example.org/> .

ex:order ex:steps ( ex:parse ex:fetch ) .



## 02.04.3 Single values

- `graph.set(triple)` keeps only one triple for its subject and predicate. Other triples with the same subject and predicate are removed.

In [5]:
sg = StarLayerGraph(prefixes={"ex": "http://example.org/"})

print("1. set adds the triple")
sg.set((EXL.order, EXL.label, literal("first")))
print(sg.serialize(format="turtle", only_used=True))

print("2. set again replaces it: the triple with the first label is removed")
sg.set((EXL.order, EXL.label, literal("second")))
print(sg.serialize(format="turtle", only_used=True))

1. set adds the triple
@prefix ex: <http://example.org/> .
ex:order ex:label "first" .

2. set again replaces it: the triple with the first label is removed
@prefix ex: <http://example.org/> .
ex:order ex:label "second" .



## 02.04.4 Walks

`graph.transitive_objects(subject, predicate)` and `graph.transitive_subjects(predicate, object)` follow a predicate until nothing new is reached. They return a generator of RDF terms, which can be read only once.  Wrap the generator in `list(...)` to create a list of the terms. 

However many paths reach a term, it is returned once. The starting term comes first. A term that is met again is skipped, which is also why a cycle always ends.

In [6]:
from starlayer.graph import RDFS

wg = StarLayerGraph()

# Dog is a Mammal, a Mammal is an Animal, and an Animal is a Dog: a cycle
for subClass, superClass in [(EXL.Dog, EXL.Mammal), (EXL.Mammal, EXL.Animal), (EXL.Animal, EXL.Dog)]:
    wg.add((subClass, RDFS.subClassOf, superClass))

print("transitive_objects and transitive_subjects follow a predicate, and a cycle ends")
# each call returns a generator: the list comprehension reads it once and keeps the terms as text
# from Dog, follow rdfs:subClassOf upwards: Dog, then Mammal, then Animal, and back to Dog (not repeated)
print("  from Dog, up:     ", [str(n) for n in wg.transitive_objects(EXL.Dog, RDFS.subClassOf)])
# from Animal, follow rdfs:subClassOf downwards: the terms that reach Animal
print("  from Animal, down:", [str(n) for n in wg.transitive_subjects(RDFS.subClassOf, EXL.Animal)])

print("\nA generator can be read only once. Wrap it in list(...) to keep the terms")
kept = list(wg.transitive_objects(EXL.Dog, RDFS.subClassOf))  # list(...) reads every term and keeps them
print( kept)


transitive_objects and transitive_subjects follow a predicate, and a cycle ends
  from Dog, up:      ['<http://example.org/Dog>', '<http://example.org/Mammal>', '<http://example.org/Animal>']
  from Animal, down: ['<http://example.org/Animal>', '<http://example.org/Mammal>', '<http://example.org/Dog>']

A generator can be read only once. Wrap it in list(...) to keep the terms
[<NamedNode value=http://example.org/Dog>, <NamedNode value=http://example.org/Mammal>, <NamedNode value=http://example.org/Animal>]


## 02.04.5 Working with Prefixes

An IRI is a global name for a resource, so it tends to be long and difficult to work with, such as `http://example.org/alice`.

Turtle, SPARQL and other text formats support prefixes, which let you write a short prefixed name such as `ex:alice` instead of the long IRI.

A prefix is a short name such as `ex` that stands for a namespace, an IRI such as `http://example.org/`. A prefixed name consists of a prefix and a local name, so `ex:alice` stands for the full IRI `http://example.org/alice`. Prefixes are shorthand for reading and writing text and for querying graphs. The graph always stores full IRIs, so adding or removing a prefix never changes the triples.

A new graph starts with no prefixes. There are various ways to add prefixes to a graph or dataset, and to remove them.

- `StarLayerGraph(prefixes={...})` gives a graph a set of prefixes when created.
- `graph.bind(prefix, namespace)` adds a prefix (`override=True` replaces a prefix that is already bound).
- `graph.parse(...)` adds the prefixes that the document declares.
- `graph.unbind(prefix)` removes one.
- `set_default_prefixes({...})` changes what every new graph starts with (`None` clears it). `DEFAULT_PREFIXES` shows the current default prefixes. This is a setting for the whole process: it applies to every graph created afterwards.


In [7]:
from starlayer.graph import DEFAULT_PREFIXES, set_default_prefixes

print("1. when the graph is made")
pf = StarLayerGraph(prefixes={"ex1": "http://example1.org/", "ex2": "http://example2.org/"})
print("  ", sorted(pf.namespaces()))

print("2. bind")
pf.bind("foaf", "http://xmlns.com/foaf/0.1/")
print("  ", sorted(pf.namespaces()))

print("3. parse adds the prefixes the document declares")
pf.parse(data='@prefix dc: <http://purl.org/dc/terms/> . <http://example.org/a> dc:title "T" .', format="turtle")
print("  ", sorted(pf.namespaces()))

print("4. unbind removes one")
print("  ", pf.unbind("foaf"), "->", sorted(pf.namespaces()))

print("5. set_default_prefixes changes what every new graph starts with")
set_default_prefixes({"ex": "http://example.org/"})
print("   default prefixes: ", DEFAULT_PREFIXES)
pf1 = StarLayerGraph()
print("   a new graph starts with:", sorted(pf1.namespaces()))
set_default_prefixes(None)
print("   after clearing:", DEFAULT_PREFIXES)

1. when the graph is made
   [('ex1', 'http://example1.org/'), ('ex2', 'http://example2.org/')]
2. bind
   [('ex1', 'http://example1.org/'), ('ex2', 'http://example2.org/'), ('foaf', 'http://xmlns.com/foaf/0.1/')]
3. parse adds the prefixes the document declares
   [('dc', 'http://purl.org/dc/terms/'), ('ex1', 'http://example1.org/'), ('ex2', 'http://example2.org/'), ('foaf', 'http://xmlns.com/foaf/0.1/')]
4. unbind removes one
   True -> [('dc', 'http://purl.org/dc/terms/'), ('ex1', 'http://example1.org/'), ('ex2', 'http://example2.org/')]
5. set_default_prefixes changes what every new graph starts with
   default prefixes:  {'ex': 'http://example.org/'}
   a new graph starts with: [('ex', 'http://example.org/')]
   after clearing: {}


### Using prefixes

Serializing, queries and updates use the graph's prefixes. Parsing works the other way: a document must declare any prefix it uses, and the graph then binds the prefixes that the document declares.

- Serialization formats that support prefixes (e.g. Turtle) write prefixed names when a prefix is bound. `serialize(only_used=True)` declares only the prefixes that the output uses.
- Queries and updates can use prefixed names bound in the graph without declaring the prefixes inside the query.

Other prefix related methods include:

- `graph.namespaces()` lists the bound prefixes.
- `graph.used_prefixes()` lists the prefixes that the graph's triples actually use.
- `graph.expand_curie("ex:alice")` returns the IRI that a prefixed name stands for.
- `graph.compute_qname(iri)` splits an IRI into `(prefix, namespace, local name)`. If no bound prefix fits, it makes one (`ns1`, `ns2`, ...) and binds it to the graph. With `generate=False` it binds nothing and raises `KeyError` for missing prefixes.
- `graph.qname_term(term)` returns the Turtle text of an RDF term, using the graph's prefixes

In [8]:
# graph is created with a set of prefixes
ug = StarLayerGraph(prefixes={"ex": "http://example.org/", "foaf": "http://xmlns.com/foaf/0.1/", "dc": "http://purl.org/dc/terms/"})

# graph is populated from a document using parse. The document uses full IRIs: a Turtle document has to declare its own prefixes, and the graph's prefixes are not applied to it.
ug.parse(data='<http://example.org/alice> <http://xmlns.com/foaf/0.1/name> "Alice" .', format="turtle")

print("1. bound prefixes, and the ones the triples use")
print("  bound:", sorted(ug.namespaces()))
print("  used: ", ug.used_prefixes())

print("\n2. a prefixed name to an IRI, and an IRI to its parts")
print("  ", ug.expand_curie("ex:alice"))
print("  ", ug.compute_qname(NamedNode("http://example.org/alice")))

print("\n3. generate=False: a KeyError when no bound prefix is found")
try:
    ug.compute_qname(NamedNode("http://unknown.example/x#y"), generate=False)
except KeyError as error:
    print("   KeyError:", error)

print("\n4. generate=True (the default): a prefix is made and bound to the graph")
print("  ", ug.compute_qname(NamedNode("http://unknown.example/x#y")))
print("   bound now:", sorted(prefix for prefix, namespace in ug.namespaces()))

print("\n5. a term as Turtle text")
print("  ", ug.qname_term(NamedNode("http://example.org/alice")))

print("\n6. a query with no PREFIX line")
print("  ", [str(row[0].value) for row in ug.query("SELECT ?name WHERE { ex:alice foaf:name ?name }")])

print("\n7. serialize with only the prefixes it needs")
print(ug.serialize(format="turtle", only_used=True))


1. bound prefixes, and the ones the triples use
  bound: [('dc', 'http://purl.org/dc/terms/'), ('ex', 'http://example.org/'), ('foaf', 'http://xmlns.com/foaf/0.1/')]
  used:  [('ex', 'http://example.org/'), ('foaf', 'http://xmlns.com/foaf/0.1/')]

2. a prefixed name to an IRI, and an IRI to its parts
   <http://example.org/alice>
   ('ex', 'http://example.org/', 'alice')

3. generate=False: a KeyError when no bound prefix is found
   KeyError: "no prefix is bound for 'http://unknown.example/x#'"

4. generate=True (the default): a prefix is made and bound to the graph
   ('ns1', 'http://unknown.example/x#', 'y')
   bound now: ['dc', 'ex', 'foaf', 'ns1']

5. a term as Turtle text
   ex:alice

6. a query with no PREFIX line
   ['Alice']

7. serialize with only the prefixes it needs
@prefix foaf: <http://xmlns.com/foaf/0.1/> .
@prefix ex: <http://example.org/> .
ex:alice foaf:name "Alice" .



## 02.04.6 What a graph contains

`graph.profile()` and `dataset.profile()` (or `profile(graph)` and `profile(dataset)`) read a graph, or a dataset, once and report what it contains:

- A dataset is profiled as one graph, the union of all its graphs, `profile.graphs` contains the profile of each graph of a dataset.

A profile has:

- `classes`: each class with its number of instances.
- `properties`: each predicate (an IRI) with its number of triples.
- `instances(cls)` lists the instances of a class.
- `class_properties(cls)`, and `properties_of(nodes)` for any set of subjects, give the properties those subjects use as `PropertyUse(property, nodes, max_values, datatypes, kinds)`: how many of the subjects have the property (`nodes`), the most values one subject has, the datatypes of the literals it has as values, and the kinds of RDF term it has as values (`"iri"`, `"blank"`, `"literal"`, `"triple"`).

In [14]:
from starlayer.graph import RDF, XSD

# a graph with two people: Ann has four properties, and Bob has a name and a second name
# (the prefixes are bound so that qname_term can print ex:Person instead of the whole IRI)
pg = StarLayerGraph(prefixes={"ex": "http://example.org/", "rdf": str(RDF), "xsd": str(XSD)})
pg.parse(
    data="""
    @prefix ex: <http://example.org/> .

    ex:a a ex:Person ; ex:name "Ann" ; ex:age 30 ; ex:knows ex:b .
    ex:b a ex:Person ; ex:name "Bob", "Robert" .
    """,
    format="turtle",
)

# profile the graph once: everything below reads the profile, not the graph
profile = pg.profile()

print("1. classes: each class with its number of instances")
for cls, count in profile.classes.items():
    print("  ", pg.qname_term(cls), count)

print("\n2. class_properties: the properties used by the instances of ex:Person")
for use in profile.class_properties(EXL.Person):
    print("  ", pg.qname_term(use.property))
    print("      subjects that have it:", use.nodes)
    print("      most values for one subject:", use.max_values)
    print("      kinds of value:", sorted(use.kinds))
    print("      datatypes of its literals:", sorted(pg.qname_term(datatype) for datatype in use.datatypes))

1. classes: each class with its number of instances
   ex:Person 2

2. class_properties: the properties used by the instances of ex:Person
   ex:name
      subjects that have it: 2
      most values for one subject: 2
      kinds of value: ['literal']
      datatypes of its literals: ['xsd:string']
   rdf:type
      subjects that have it: 2
      most values for one subject: 1
      kinds of value: ['iri']
      datatypes of its literals: []
   ex:age
      subjects that have it: 1
      most values for one subject: 1
      kinds of value: ['literal']
      datatypes of its literals: ['xsd:integer']
   ex:knows
      subjects that have it: 1
      most values for one subject: 1
      kinds of value: ['iri']
      datatypes of its literals: []


In [ ]:
from starlayer.graph import StarLayerDataset

# a dataset with two graphs that hold data, and an empty one
ds = StarLayerDataset(prefixes={"ex": "http://example.org/"})
ds.graph(EXL.people).parse(
    data="""
    @prefix ex: <http://example.org/> .

    ex:ann a ex:Person ; ex:name "Ann" .
    ex:bob a ex:Person .
    """,
    format="turtle",
)
ds.graph(EXL.pets).parse(data="@prefix ex: <http://example.org/> . ex:rex a ex:Dog .", format="turtle")
ds.ox.add_graph(EXL.archive)

# the profile of a dataset is the profile of the union of its graphs
dataset_profile = ds.profile()
print("1. the dataset as a whole")
for cls, count in dataset_profile.classes.items():
    print("  ", ds.qname_term(cls), count)

# profile.graphs has the profile of each graph, by graph name
print("\n2. profile.graphs: the classes in each graph")
for name, graph_profile in dataset_profile.graphs.items():
    label = ds.qname_term(name) if isinstance(name, NamedNode) else "the default graph"
    classes = {ds.qname_term(cls): count for cls, count in graph_profile.classes.items()}
    print("  ", label, classes)

1. the dataset as a whole
   ex:Dog 1
   ex:Person 2

2. profile.graphs: the classes in each graph
   the default graph {}
   ex:archive {}
   ex:people {'ex:Person': 2}
   ex:pets {'ex:Dog': 1}
